# Feature engineering — Supplementary exploration

Author-origin quantitative analysis retained. See the English consolidated case study for validated interpretation and executive storytelling. The original explorations were produced before the portfolio restructuring and may contain Italian in analytical print messages.

## 1. Import e percorsi


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "deep_dives":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
elif PROJECT_ROOT.name.lower() in {"cleaning", "eda", "notebooks"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "output"
CLEAN_FILE = OUTPUT_DIR / "adidas_sales_clean.csv"
FEATURES_FILE = OUTPUT_DIR / "adidas_sales_features.csv"
if not CLEAN_FILE.is_file():
    raise FileNotFoundError("Manca il CSV pulito: esegui prima il notebook 01 — Data Cleaning")

print("Input:", CLEAN_FILE)
print("Output:", FEATURES_FILE)

## 2. Caricamento dati puliti e validazione preliminare

In [ ]:
df = pd.read_csv(
    CLEAN_FILE,
    dtype={"retailer_id": "string"},
    parse_dates=["invoice_date"],
    encoding="utf-8-sig",
)

REQUIRED_COLS = {"invoice_date", "product", "price_per_unit", "units_sold", "total_sales"}
if not REQUIRED_COLS.issubset(df.columns):
    raise ValueError(f"Colonne assenti: {sorted(REQUIRED_COLS - set(df.columns))}")
if df["invoice_date"].isna().any():
    raise ValueError("Date mancanti o illeggibili: correggere il notebook 01")

initial_rows = len(df)
print("Dati puliti caricati:", df.shape)
display(df.head(3))

## 3. Anno, mese numerico, nome del mese e anno-mese


In [ ]:
MONTH_NAMES_IT = {
    1: "Gennaio", 2: "Febbraio", 3: "Marzo", 4: "Aprile",
    5: "Maggio", 6: "Giugno", 7: "Luglio", 8: "Agosto",
    9: "Settembre", 10: "Ottobre", 11: "Novembre", 12: "Dicembre",
}

df["year"] = df["invoice_date"].dt.year.astype("int64")
df["month_number"] = df["invoice_date"].dt.month.astype("int64")
df["month_name_it"] = df["month_number"].map(MONTH_NAMES_IT).astype("string")
df["year_month"] = df["invoice_date"].dt.strftime("%Y-%m").astype("string")

display(df[["invoice_date", "year", "month_number", "month_name_it", "year_month"]].head(5))

## 4. Giorno della settimana e weekend

In [ ]:
WEEKDAY_NAMES_IT = {
    0: "Lunedì", 1: "Martedì", 2: "Mercoledì", 3: "Giovedì",
    4: "Venerdì", 5: "Sabato", 6: "Domenica",
}

df["weekday_number"] = df["invoice_date"].dt.dayofweek.astype("int64")
df["weekday_name_it"] = df["weekday_number"].map(WEEKDAY_NAMES_IT).astype("string")
df["is_weekend"] = df["weekday_number"].isin([5, 6])

display(df[["invoice_date", "weekday_number", "weekday_name_it", "is_weekend"]].head(7))

## 5. Stagione di calendario

In [ ]:
SEASON_BY_MONTH = {
    12: "Inverno", 1: "Inverno", 2: "Inverno",
    3: "Primavera", 4: "Primavera", 5: "Primavera",
    6: "Estate", 7: "Estate", 8: "Estate",
    9: "Autunno", 10: "Autunno", 11: "Autunno",
}
df["season_it"] = df["month_number"].map(SEASON_BY_MONTH).astype("string")
display(df[["invoice_date", "month_name_it", "season_it"]].head(5))

## 6. Segmento di genere del prodotto

In [ ]:
product_text = df["product"].astype("string")
product_gender_source = product_text.str.extract(r"^(Men's|Women's)\b", flags=0, expand=False)
GENDER_MAP = {"Men's": "Uomo", "Women's": "Donna"}
df["product_gender"] = (
    product_gender_source.map(GENDER_MAP).fillna("Non classificato").astype("string")
)

display(df[["product", "product_gender"]].head(8))

## 7. Categoria di prodotto (variabile aggiuntiva)


In [ ]:
is_apparel = product_text.str.contains("Apparel", case=False, na=False, regex=False)
is_footwear = product_text.str.contains("Footwear", case=False, na=False, regex=False)

df["product_category"] = pd.Series("Non classificato", index=df.index, dtype="string")
df.loc[is_apparel, "product_category"] = "Abbigliamento"
df.loc[is_footwear & ~is_apparel, "product_category"] = "Calzature"

display(df[["product", "product_gender", "product_category"]].drop_duplicates().head(10))

## 8. Controllo delle feature ed esportazione

In [ ]:
FEATURE_COLS = [
    "year", "month_number", "month_name_it", "year_month",
    "weekday_number", "weekday_name_it", "is_weekend",
    "season_it", "product_gender", "product_category",
]

assert len(df) == initial_rows, "La creazione delle feature non deve rimuovere righe"
assert not df[FEATURE_COLS].isna().any().any(), "Sono presenti feature mancanti"
assert df["month_number"].between(1, 12).all()
assert df["weekday_number"].between(0, 6).all()
assert (df["is_weekend"] == df["weekday_number"].isin([5, 6])).all()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(FEATURES_FILE, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
print("Feature engineering completato:", FEATURES_FILE)
print("Righe:", len(df), "| Colonne totali:", df.shape[1], "| Feature aggiunte:", len(FEATURE_COLS))
display(df[["invoice_date", "product", *FEATURE_COLS]].head(5))